# Explainability — Dev Log

## Objetivo e papel no pipeline

Este notebook documenta o desenvolvimento do módulo `core/explainability/`
do AthenaGov AI.

**O que o módulo faz:** fornece uma única função pública,
`explain(factors: dict[str, float], subject: str, narrative_template: str | None = None) -> ExplainabilityResult`,
que recebe um dicionário genérico de fatores numéricos (`{nome: contribuição}`)
e devolve uma explicação estruturada (`shared.schemas.ExplainabilityResult`)
com os fatores ordenados por magnitude e uma narrativa em português natural.

**MUITO IMPORTANTE — direção da dependência:** este módulo é uma
**utilidade genérica e standalone**. Ele:

- **NÃO importa** nenhum outro módulo do projeto (`trust_score`, `policy_engine`,
  `pii_detection`, `ripd_engine`, etc.) — só importa `shared.schemas.ExplainabilityResult`;
- **NÃO conhece** conceitos de domínio como "PII", "política" ou "trust score" —
  só enxerga números (`float`) e texto livre (`str`);
- é **injetado** pelos outros módulos: são eles que calculam seus próprios
  fatores de domínio e chamam `explain(...)` diretamente, usando o resultado
  como parte da própria saída (ex.: `TrustScoreResult.explanation`).

Ou seja, a seta de dependência aponta sempre **dos módulos de domínio para
este motor**, nunca o contrário. Isso é o que permite este módulo ser
desenvolvido e testado de forma 100% isolada, em paralelo com os módulos
que vão consumi-lo.

## Decisões de design

1. **Determinístico, não é machine learning.** Não há modelo treinado nem
   inferência estatística. "Importância" de um fator é simplesmente
   `abs(contribuição)`, calculada e fornecida por quem chama a função. Isso é
   deixado explícito no docstring do módulo para não confundir revisores.

2. **Ordenação por magnitude, preservando o sinal na narrativa.** Fatores são
   ordenados por `abs(valor)` decrescente, mas a narrativa sempre reporta o
   valor com sinal (`+0.25` / `-0.42`) e descreve a direção do efeito
   ("aumentando o resultado" / "reduzindo o resultado"). Fatores com
   `valor == 0.0` são tratados como neutros, sem quebrar a ordenação.

3. **`factors` no resultado preserva o dicionário original.** A ordenação por
   magnitude afeta apenas a narrativa gerada — o campo `factors` do
   `ExplainabilityResult` retorna exatamente o que foi passado, sem reordenar
   ou filtrar, para não surpreender quem consome o resultado programaticamente.

4. **`narrative_template` com dois modos, sem ambiguidade silenciosa.**
   Inicialmente a implementação tentava sempre `template.format(...)`, mas
   isso causava um bug sutil: um template *sem* placeholders (`"Resumo
   executivo customizado"`) é uma string válida para `.format()` e retorna
   inalterada — descartando silenciosamente a explicação dos fatores. A
   correção: só entra no modo "molde" (`.format(subject=..., body=...)`) se o
   template contiver literalmente `{subject}` ou `{body}`; caso contrário,
   opera em modo "prefixo", anexando a explicação padrão dos fatores ao final
   do texto fornecido. Testado explicitamente em
   `test_custom_narrative_template_as_prefix` e
   `test_custom_narrative_template_with_placeholders`.

5. **Dicionário vazio tem narrativa própria, não uma exceção.** Um módulo
   consumidor pode legitimamente não ter fatores a reportar (ex.: decisão sem
   nenhuma regra acionada); a função retorna uma narrativa explicando a
   ausência de fatores em vez de lançar erro.

6. **`subject` é texto livre, sem validação de formato.** O motor não impõe
   vocabulário sobre o que está sendo explicado — cabe a quem chama definir a
   convenção (ex.: `"trust_score"`, `"policy_decision:pol_health_data"`).

## Setup

In [1]:
import sys
from pathlib import Path

# notebook roda de dentro de notebooks/; adiciona a raiz do repo ao sys.path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from core.explainability.engine import explain
from shared.schemas import ExplainabilityResult

print("Módulo carregado com sucesso.")
print("Assinatura pública:")
print(
    "explain(factors: dict[str, float], subject: str, "
    "narrative_template: str | None = None) -> ExplainabilityResult"
)

Módulo carregado com sucesso.
Assinatura pública:
explain(factors: dict[str, float], subject: str, narrative_template: str | None = None) -> ExplainabilityResult


## Exemplo 1 — simulação de `trust_score`

Fatores **ilustrativos** (inventados apenas para esta demo) que o módulo
`trust_score` (desenvolvido por outro agente) poderia calcular e injetar
neste motor. Este notebook não importa `trust_score` — os valores abaixo só
simulam o formato de entrada que ele produziria.

In [2]:
# Exemplo 1 — ilustrativo: fatores hipotéticos que o módulo trust_score
# poderia calcular e injetar aqui. Valores INVENTADOS para fins de demo.
trust_score_factors = {
    "historico_incidentes_seguranca": -0.42,
    "base_legal_valida": 0.25,
    "consentimento_explicito": 0.18,
    "criptografia_em_repouso": 0.10,
    "ausencia_auditoria_recente": -0.15,
}

result_trust = explain(
    factors=trust_score_factors,
    subject="trust_score",
)

print("subject:", result_trust.subject)
print("factors:", result_trust.factors)
print()
print("narrative:")
print(result_trust.narrative)

subject: trust_score
factors: {'historico_incidentes_seguranca': -0.42, 'base_legal_valida': 0.25, 'consentimento_explicito': 0.18, 'criptografia_em_repouso': 0.1, 'ausencia_auditoria_recente': -0.15}

narrative:
A decisão sobre 'trust_score' foi principalmente influenciada por 'historico_incidentes_seguranca' (contribuição de -0.42, reduzindo o resultado); seguido por 'base_legal_valida' (contribuição de +0.25, aumentando o resultado); seguido por 'consentimento_explicito' (contribuição de +0.18, aumentando o resultado); seguido por 'ausencia_auditoria_recente' (contribuição de -0.15, reduzindo o resultado); seguido por 'criptografia_em_repouso' (contribuição de +0.10, aumentando o resultado).


## Exemplo 2 — simulação de `policy_decision` (com `narrative_template` customizado)

Fatores **ilustrativos** de uma decisão hipotética do `policy_engine` para
uma política de dado de saúde, usando um `narrative_template` no modo
"molde" (`{body}`).

In [3]:
# Exemplo 2 — ilustrativo: fatores hipotéticos de uma decisão do policy_engine
# para uma política de dado de saúde. Valores INVENTADOS para fins de demo.
policy_factors = {
    "categoria_dado_sensivel_saude": -0.55,
    "finalidade_pesquisa_cientifica": 0.20,
    "menor_de_idade_envolvido": -0.30,
    "anonimizacao_aplicada": 0.35,
}

result_policy = explain(
    factors=policy_factors,
    subject="policy_decision:pol_health_data",
    narrative_template="[Parecer automático] {body}. Revisão humana recomendada.",
)

print("subject:", result_policy.subject)
print("factors:", result_policy.factors)
print()
print("narrative:")
print(result_policy.narrative)

subject: policy_decision:pol_health_data
factors: {'categoria_dado_sensivel_saude': -0.55, 'finalidade_pesquisa_cientifica': 0.2, 'menor_de_idade_envolvido': -0.3, 'anonimizacao_aplicada': 0.35}

narrative:
[Parecer automático] principalmente influenciada por 'categoria_dado_sensivel_saude' (contribuição de -0.55, reduzindo o resultado); seguido por 'anonimizacao_aplicada' (contribuição de +0.35, aumentando o resultado); seguido por 'menor_de_idade_envolvido' (contribuição de -0.30, reduzindo o resultado); seguido por 'finalidade_pesquisa_cientifica' (contribuição de +0.20, aumentando o resultado). Revisão humana recomendada.


## Exemplo 3 — simulação de `pii_detection`

Fatores **ilustrativos** que o módulo `pii_detection` poderia usar para
explicar por que um trecho de texto foi classificado como alto risco.

In [4]:
# Exemplo 3 — ilustrativo: fatores hipotéticos que o pii_detection poderia
# usar para explicar por que um trecho foi marcado como alto risco.
# Valores INVENTADOS para fins de demo.
pii_factors = {
    "cpf_detectado_alta_confianca": 0.60,
    "email_detectado_media_confianca": 0.25,
    "contexto_anonimizado_parcial": -0.10,
}

result_pii = explain(
    factors=pii_factors,
    subject="pii_detection:documento_123",
)

print("subject:", result_pii.subject)
print("factors:", result_pii.factors)
print()
print("narrative:")
print(result_pii.narrative)

subject: pii_detection:documento_123
factors: {'cpf_detectado_alta_confianca': 0.6, 'email_detectado_media_confianca': 0.25, 'contexto_anonimizado_parcial': -0.1}

narrative:
A decisão sobre 'pii_detection:documento_123' foi principalmente influenciada por 'cpf_detectado_alta_confianca' (contribuição de +0.60, aumentando o resultado); seguido por 'email_detectado_media_confianca' (contribuição de +0.25, aumentando o resultado); seguido por 'contexto_anonimizado_parcial' (contribuição de -0.10, reduzindo o resultado).


## Caso de borda — dicionário de fatores vazio

In [5]:
# Caso de borda: nenhum fator disponível
result_empty = explain(factors={}, subject="trust_score")
print(result_empty.narrative)

Não há fatores registrados para explicar a decisão sobre 'trust_score'.


## Suíte de testes — execução real via subprocess

In [6]:
import subprocess

python_exe = r"C:\Users\Yuri_\.venvs\athenagov-ai\Scripts\python.exe"
proc = subprocess.run(
    [python_exe, "-m", "pytest", "core/explainability/tests", "-v"],
    cwd=str(ROOT),
    capture_output=True,
    text=True,
)
print(proc.stdout[-4000:])
if proc.stderr.strip():
    print(proc.stderr[-2000:])
print("returncode:", proc.returncode)

============================= test session starts =============================
platform win32 -- Python 3.10.8, pytest-9.1.1, pluggy-1.6.0 -- C:\Users\Yuri_\.venvs\athenagov-ai\Scripts\python.exe
cachedir: .pytest_cache
rootdir: g:\Outros computadores\Meu computador\Controle Base\Projetos, Robos e Automação\Projetos Git\Projetos Extras (Portfolio)\LGPD e IA (Terminar)
plugins: anyio-4.14.2, cov-7.1.0
collecting ... collected 12 items

core/explainability/tests/test_engine.py::test_returns_explainability_result_instance PASSED [  8%]
core/explainability/tests/test_engine.py::test_empty_factors_returns_appropriate_narrative PASSED [ 16%]
core/explainability/tests/test_engine.py::test_single_factor PASSED      [ 25%]
core/explainability/tests/test_engine.py::test_only_positive_factors PASSED [ 33%]
core/explainability/tests/test_engine.py::test_only_negative_factors PASSED [ 41%]
core/explainability/tests/test_engine.py::test_mixed_positive_and_negative_factors PASSED [ 50%]
core/explain

## Handoff Summary

### Capacidades entregues

- Motor de explicabilidade **genérico e determinístico** (não é machine
  learning) para gerar explicações em linguagem natural a partir de
  fatores numéricos com sinal.
- Ordenação de fatores por magnitude de contribuição (`abs(valor)`,
  decrescente), preservando o sinal e a direção do efeito na narrativa.
- Tratamento correto de: fatores só positivos, só negativos, mistos,
  dicionário vazio, fator único, muitos fatores, e valor neutro (`0.0`).
- Suporte a `narrative_template` customizado em dois modos (molde com
  `{subject}`/`{body}`, ou prefixo livre).
- **Zero acoplamento** com outros módulos do projeto — importa apenas
  `shared.schemas.ExplainabilityResult`.
- 12 testes pytest, todos passando (`core/explainability/tests/test_engine.py`).

### Assinatura pública exata

```python
def explain(
    factors: dict[str, float],
    subject: str,
    narrative_template: str | None = None,
) -> ExplainabilityResult:
    ...
```

Importar de: `from core.explainability.engine import explain` (ou
`from core.explainability import explain`, reexportado em `__init__.py`).

### Como outros módulos devem integrar (injeção de dependência)

O módulo consumidor **calcula seus próprios fatores de domínio** e chama
`explain(...)` diretamente — sem adapters, sem herança, sem registro. Exemplo
de como `trust_score` integraria isto (código ilustrativo, não faz parte
deste módulo):

```python
# dentro de core/trust_score/engine.py (outro módulo, outro agente)
from core.explainability.engine import explain
from shared.schemas import TrustScoreResult, RiskLevel

def compute_trust_score(...) -> TrustScoreResult:
    components = {
        "historico_incidentes_seguranca": -0.42,
        "base_legal_valida": 0.25,
        # ... calculado pela lógica de domínio do trust_score
    }
    score = 100.0 + sum(components.values()) * 100  # lógica própria do módulo

    explanation = explain(
        factors=components,
        subject="trust_score",
    )

    return TrustScoreResult(
        score=score,
        risk_level=RiskLevel.MEDIUM,
        components=components,
        explanation=explanation,  # ExplainabilityResult encaixa direto no contrato
    )
```

O mesmo padrão vale para `policy_engine` (`subject="policy_decision:<policy_id>"`)
e `ripd_engine`.

### Limitações conhecidas

- Narrativa é sempre em português e usa um template textual fixo (com
  variação apenas via `narrative_template`) — não há geração via LLM nem
  suporte a outros idiomas.
- Não há normalização/escala dos valores de `factors`: a magnitude relativa
  é interpretada literalmente pelo valor recebido, então módulos consumidores
  devem manter uma escala consistente entre si (ex.: sempre no intervalo
  aproximado [-1, 1]) para a narrativa fazer sentido.
- Empates exatos em `abs(valor)` são desempatados pela ordem de inserção do
  dicionário Python (comportamento estável do `sorted`), não por nenhuma
  regra de negócio.
- Não versiona nem persiste explicações — cada chamada é stateless; cabe ao
  chamador decidir se/como armazenar o `ExplainabilityResult` (ex.: via
  `audit_logs`).

### O que fica para V2

- Templates de narrativa mais ricos (ex.: agrupar fatores por categoria,
  destacar apenas os top-N fatores).
- Suporte a i18n (múltiplos idiomas de narrativa).
- Explicações contrafactuais ("se `fator_x` fosse diferente, o resultado
  mudaria para...") — mencionado no roadmap V2 como possível evolução ligada
  a Fairness Audit / AI Observability, mas fora do escopo determinístico
  simples do V1.